In [1]:
!pip install -q transformers pymupdf python-docx pillow opencv-python numpy

import os, re, fitz, torch, numpy as np
from PIL import Image
from collections import Counter, defaultdict
from google.colab import drive
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.text import WD_TAB_ALIGNMENT, WD_TAB_LEADER
from docx.enum.style import WD_STYLE_TYPE

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/GOST_Project/layoutlm_model-20260629T015048Z-3-001/layoutlm_model/checkpoint-498"
INPUT_PDF = "/content/drive/MyDrive/GOST_Project/Субботина_текст_поломанный.pdf"
OUTPUT_DOCX = "/content/drive/MyDrive/GOST_Project/smart_result2.docx"

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = {0: 'BODY', 1: 'FIGURE', 2: 'FIGURE_CAPTION', 3: 'FOOTNOTE', 4: 'FORMULA',
            5: 'OTHER', 6: 'REFERENCE', 7: 'SECTION_HEADER', 8: 'TABLE',
            9: 'TABLE_CAPTION', 10: 'TITLE', 11: 'TOC'}


def extract_pymu_data(pdf_path):
    doc = fitz.open(pdf_path)
    all_pages = {}

    for page_num in range(len(doc)):
        page = doc[page_num]
        words_data = page.get_text("words")
        page_words = []

        page_rect = page.rect
        page_width = page_rect.width
        page_height = page_rect.height

        for w in words_data:
            x0 = int(w[0] / page_width * 1000)
            y0 = int(w[1] / page_height * 1000)
            x1 = int(w[2] / page_width * 1000)
            y1 = int(w[3] / page_height * 1000)

            page_words.append({
                "text": w[4],
                "box": [x0, y0, x1, y1]
            })

        all_pages[page_num + 1] = page_words

    return all_pages

print("Извлекаем текст из PDF через PyMuPDF")
pymu_data = extract_pymu_data(INPUT_PDF)
print(f"Извлечено {len(pymu_data)} страниц")


def predict_page_pymu(page_num):
    words_data = pymu_data.get(page_num, [])
    if not words_data:
        return []

    words = [item["text"] for item in words_data]
    boxes = [item["box"] for item in words_data]

    image = Image.new('RGB', (1000, 1000), color='white')

    encoding = processor(image, words, boxes=boxes, truncation=True,
                         padding="max_length", max_length=512, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**encoding)

    preds = outputs.logits.argmax(-1)[0]
    word_ids = encoding.word_ids()

    result = []
    used = set()
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx in used:
            continue
        used.add(word_idx)
        if word_idx < len(words):
            result.append({
                "text": words[word_idx],
                "label": id2label[preds[token_idx].item()],
                "box": boxes[word_idx]
            })

    return result

def build_lines(predictions, y_threshold=18):
    if not predictions:
        return []

    predictions = sorted(predictions, key=lambda x: (x["box"][1], x["box"][0]))
    lines, current_line, prev_y = [], [], None

    for item in predictions:
        y_top = item["box"][1]
        if prev_y is None or abs(y_top - prev_y) > y_threshold:
            if current_line:
                lines.append(current_line)
            current_line = [item]
        else:
            current_line.append(item)
        prev_y = y_top

    if current_line:
        lines.append(current_line)

    result = []
    for line in lines:
        line = sorted(line, key=lambda x: x["box"][0])
        labels = [item["label"] for item in line]
        main_label = Counter(labels).most_common(1)[0][0]
        result.append({
            "text": " ".join(item["text"] for item in line).strip(),
            "label": main_label,
            "box": [
                min(item["box"][0] for item in line),
                min(item["box"][1] for item in line),
                max(item["box"][2] for item in line),
                max(item["box"][3] for item in line)
            ]
        })

    return result

print("Извлекаем кандидатов из модели")
all_toc_candidates, all_ref_candidates = [], []
pages_data = {}

for page_num in sorted(pymu_data.keys()):
    predictions = predict_page_pymu(page_num)
    lines = build_lines(predictions)
    pages_data[page_num] = lines

    for line in lines:
        if line["label"] == "TOC":
            all_toc_candidates.append({**line, "page": page_num})
        elif line["label"] == "REFERENCE":
            all_ref_candidates.append({**line, "page": page_num})

print(f"Кандидатов TOC: {len(all_toc_candidates)}")
print(f"Кандидатов REFERENCE: {len(all_ref_candidates)}")


def is_toc_heading(text):
    return text.strip().lower() in ["содержание", "оглавление"]

def is_ref_heading(text):
    return text.strip().lower() in ["список литературы", "библиографический список", "библиография", "список использованных источников"]

def filter_toc(candidates, pages_data):
    """
    Собирает содержание со страницы оглавления.
    """
    result = []

    toc_page = None
    for c in candidates:
        if is_toc_heading(c["text"]):
            toc_page = c["page"]
            break

    if toc_page is None:
        toc_page = 2

    page = toc_page
    max_pages = min(toc_page + 5, max(pages_data.keys()) + 1)

    while page <= max_pages and page in pages_data:
        for line in pages_data[page]:
            txt = line["text"].strip()
            if not txt:
                continue

            if txt.upper().startswith("СПИСОК ЛИТЕРАТУРЫ"):
                return result
            if txt.upper().startswith("ЗАКЛЮЧЕНИЕ"):
                result.append({**line, "page": page})
                return result

            is_toc_line = False

            if re.match(r'^\s*(\d+(\.\d+)*\.?\s+)', txt):
                is_toc_line = True

            elif re.match(r'^\s*[Гг]лава\s+\d+', txt):
                is_toc_line = True
            elif re.match(r'^\s*[Рр]аздел\s+\d+', txt):
                is_toc_line = True

            elif txt.upper() in ["АННОТАЦИЯ", "ВВЕДЕНИЕ", "ЗАКЛЮЧЕНИЕ", "СПИСОК ТЕРМИНОВ И АББРЕВИАТУР"]:
                is_toc_line = True

            elif re.match(r'^[А-ЯA-Z]', txt) and re.search(r'\d+\s*$', txt):
                is_toc_line = True

            elif re.search(r'\d+\s*$', txt):
                if len(txt) < 100:
                    is_toc_line = True

            if is_toc_line:
                result.append({**line, "page": page})

        page += 1

    if not result:
        return candidates

    return result

def filter_reference(candidates, pages_data):
    """Фильтрует библиографию"""
    if not candidates:
        return []

    ref_start_page = None
    for item in candidates:
        if is_ref_heading(item["text"]):
            ref_start_page = item["page"]
            break

    if ref_start_page is None:
        ref_start_page = max(item["page"] for item in candidates)

    result = []
    for item in candidates:
        if item["page"] < ref_start_page - 1:
            continue
        if item["page"] > ref_start_page + 5:
            continue
        result.append(item)

    if not result:
        max_page = max(pages_data.keys())
        for page_num in range(max_page - 3, max_page + 1):
            if page_num not in pages_data:
                continue
            for line in pages_data[page_num]:
                if line["label"] == "REFERENCE":
                    result.append({**line, "page": page_num})

    return result

print("\nПрименяем логический фильтр к TOC")
filtered_toc = filter_toc(all_toc_candidates, pages_data)

print("\nПрименяем логический фильтр к REFERENCE")
filtered_ref = filter_reference(all_ref_candidates, pages_data)


def clean_toc_entries(toc_items):
    """Очищает TOC от мусора"""
    cleaned = []
    for item in toc_items:
        text = item["text"].strip()

        if len(text) < 5:
            continue

        if re.search(r'[Оо]пределени[еяй]|[Яя]вляют[сся]|[Нн]е являют[сся]', text):
            continue

        has_section = re.match(r'^\s*(\d+(\.\d+)*\.?\s+)', text) is not None
        has_page = re.search(r'\d+\s*$', text) is not None

        if has_section or text.upper() in ["ВВЕДЕНИЕ", "ЗАКЛЮЧЕНИЕ", "СПИСОК ТЕРМИНОВ И АББРЕВИАТУР", "АННОТАЦИЯ"]:
            if has_section and has_page:
                cleaned.append(item)
            elif has_section and len(text) < 80:
                cleaned.append(item)
            elif text.upper() in ["ВВЕДЕНИЕ", "ЗАКЛЮЧЕНИЕ", "СПИСОК ТЕРМИНОВ И АББРЕВИАТУР", "АННОТАЦИЯ"]:
                if has_page:
                    cleaned.append(item)

    return cleaned


def split_reference_entries(ref_items):
    """Разбивает библиографию на записи — УНИВЕРСАЛЬНАЯ ВЕРСИЯ"""
    ref_items = sorted(ref_items, key=lambda x: (x["page"], x["box"][1]))

    lines = []
    for item in ref_items:
        txt = item["text"].strip()
        if not txt:
            continue
        if is_ref_heading(txt):
            continue
        if len(txt) < 10:
            continue
        lines.append(txt)

    CONTINUATION_WORDS = {
        "Conference", "Proceedings", "NeurIPS", "ICML", "NIPS",
        "Computation", "Processing", "Journal", "Review", "Advances",
        "arXiv:", "https://", "http://", "doi.org", "pp.", "vol.",
        "and", "of", "for", "with", "from", "at", "the", "in", "on",
        "Lecture", "Notes", "Tools", "Algorithms", "Construction",
        "Analysis", "Symposium", "Workshop", "Meeting"
    }

    entries = []
    current = []

    for line in lines:
        is_continuation = False
        for word in CONTINUATION_WORDS:
            if line.startswith(word):
                is_continuation = True
                break

        if re.search(r'arXiv:|https://|http://|doi.org', line):
            is_continuation = True

        is_new = False

        if re.match(r'^[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё\-]+[,;]', line):
            is_new = True
        elif re.match(r'^[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё\-]+\s+[A-ZА-ЯЁ]\.', line):
            is_new = True
        elif re.match(r'^\d+[\.\)]', line) or re.match(r'^\[\d+\]', line):
            is_new = True
        elif re.match(r'^[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё\-]+\s+\d{4}', line):
            is_new = True

        if is_continuation and current:
            current.append(line)
        elif is_new:
            if current:
                entries.append(" ".join(current))
            current = [line]
        else:
            if current:
                current.append(line)
            else:
                current = [line]

    if current:
        entries.append(" ".join(current))

    merged = []
    i = 0
    while i < len(entries):
        if i + 1 < len(entries):
            if not entries[i].endswith('.') or len(entries[i]) < 50:
                if not re.match(r'^[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё\-]+[,;]', entries[i+1]):
                    merged.append(entries[i] + " " + entries[i+1])
                    i += 2
                    continue
        merged.append(entries[i])
        i += 1

    seen = set()
    unique = []
    for entry in merged:
        entry = re.sub(r'^[\s,;]+', '', entry)
        entry = re.sub(r'\s+', ' ', entry).strip()

        if len(entry) < 20:
            continue
        if re.match(r'^[\d\s\.]+$', entry):
            continue

        if entry not in seen:
            seen.add(entry)
            unique.append(entry)

    return unique


def render_toc(doc, toc_items):
    if not toc_items:
        return

    toc_items = clean_toc_entries(toc_items)
    toc_items = sorted(toc_items, key=lambda x: (x["page"], x["box"][1]))

    p = doc.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    run = p.add_run("СОДЕРЖАНИЕ")
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(14)
    p.paragraph_format.space_after = Pt(12)

    style = doc.styles.add_style('TOCStyle', 1)
    style.font.name = 'Times New Roman'
    style.font.size = Pt(14)
    style.paragraph_format.space_after = Pt(0)
    style.paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    tab_stop = Cm(14.5)
    style.paragraph_format.tab_stops.add_tab_stop(tab_stop, WD_TAB_ALIGNMENT.RIGHT, WD_TAB_LEADER.DOTS)

    for item in toc_items:
        text = item["text"].strip()

        if len(text) < 3:
            continue
        if is_toc_heading(text):
            continue
        if text.upper().startswith("СПИСОК ЛИТЕРАТУРЫ"):
            continue

        match = re.search(r'(.*?)(\d+)$', text)
        if match:
            title = match.group(1).strip()
            page = match.group(2)
            title = re.sub(r'\.{2,}', '', title)
            title = re.sub(r'\s+', ' ', title)
            p = doc.add_paragraph(style=style)
            p.add_run(title + "\t" + page)
        else:
            p = doc.add_paragraph(style=style)
            p.add_run(text)

def render_references(doc, ref_items):
    entries = split_reference_entries(ref_items)

    if not entries:
        return

    p = doc.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    run = p.add_run("СПИСОК ЛИТЕРАТУРЫ")
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(14)
    p.paragraph_format.space_after = Pt(12)

    for i, entry in enumerate(entries, 1):
        entry = re.sub(r'^\s*\d+[\.\)]\s*', '', entry)

        p = doc.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.first_line_indent = Cm(0)
        p.paragraph_format.space_after = Pt(6)

        run = p.add_run(f"{i}. {entry}")
        run.font.name = "Times New Roman"
        run.font.size = Pt(14)

doc = Document()
section = doc.sections[0]
section.top_margin = Cm(2)
section.bottom_margin = Cm(2)
section.left_margin = Cm(3)
section.right_margin = Cm(1.5)

render_toc(doc, filtered_toc)
if filtered_toc and filtered_ref:
    doc.add_page_break()
render_references(doc, filtered_ref)

doc.save(OUTPUT_DOCX)
print(f"\nФайл сохранён: {OUTPUT_DOCX}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.2 MB/s eta 0:00:00
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/216 [00:01<?, ?it/s]

Извлекаем текст из PDF через PyMuPDF
Извлечено 56 страниц
Извлекаем кандидатов из модели
Кандидатов TOC: 24
Кандидатов REFERENCE: 95

Применяем логический фильтр к TOC

Применяем логический фильтр к REFERENCE

Файл сохранён: /content/drive/MyDrive/GOST_Project/smart_result2.docx
